# Kaggle Setup and Usage Guide

## 1. Running Kaggle Docker Locally 

The built Docker images for the Kaggle Python environment (from [kaggle/docker-python](https://github.com/Kaggle/docker-python)),which comes with hundreds of pre-installed packages, are **hosted on Google Container Registry (GCR)**, not Docker Hub or ghcr.io.

**Official image locations:**
- **CPU-only:** `gcr.io/kaggle-images/python`
- **GPU:** `gcr.io/kaggle-gpu-images/python`



**Important notes:**
- Legacy image on Docker Hub ([hub.docker.com/r/kaggle/python](https://hub.docker.com/r/kaggle/python)) is **stale/outdated**. Use the gcr.io images above.
- Tags: Check [releases/tags](https://github.com/Kaggle/docker-python/tags) on GitHub for version-specific tags.


pre-installed packages:

- **Deep Learning:** `torch`, `tensorflow`, `keras`, `transformers`, `timm`
- **Data Science:** `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`
- **Computer Vision:** `opencv-python`, `Pillow`, `albumentations`
- **NLP:** `nltk`, `spacy`, `gensim`
- **Utilities:** `tqdm`, `requests`, `beautifulsoup4`, `plotly`

**Full list:** [kaggle_requirements.txt](https://github.com/Kaggle/docker-python/blob/main/kaggle_requirements.txt)



---

**Useful commands:**

| Action | Command |
|--------|----------|
| View logs | `docker logs --tail 50 temp-kaggle-notebook` |
| Stop | `docker stop temp-kaggle-notebook` |
| Remove container | `docker rm temp-kaggle-notebook` |
| Remove image (~45GB) | `docker rmi gcr.io/kaggle-gpu-images/python:latest` |

**Cleanup:**
```bash
docker container prune -f   # Clean stopped containers
docker image prune -f      # Clean dangling images
```


### Running Kaggle Notebooks Locally

**Pull and run:**

```bash
docker pull gcr.io/kaggle-gpu-images/python
docker stop my-kaggle
```


```bash

# Run with bash first, then manually start jupyter
docker run --rm \
    --gpus all \
    --name my-kaggle \
    -p 8888:8888 \
    -v /home/$USER/anaconda3/envs/PyTorchTutorial/src/projects:/work \
    -w /work \
    -it \
    gcr.io/kaggle-gpu-images/python:latest \
    /bin/bash
```

Then inside the container, run:
```bash
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root --ServerApp.token='kaggle123'
```


 -it gcr.io/kaggle-gpu-images/python:latest 


After starting, open `http://localhost:8888/lab?token=kaggle123`




**Example:**
```python
# In Kaggle notebooks, these are already available:
import torch
# !pip install -q some-new-package
```

---

### Alternate: Build your Customized Kaggle-Compatible Docker Image

**1. Base image:** Built on Kaggle's official GPU image:
```dockerfile
FROM gcr.io/kaggle-gpu-images/python
```

**2. Build locally:**
```bash
cd /home/$USERNAME/anaconda3/envs/PyTorchTutorial/src
docker build -t kaggle-universal:latest .
```

**3. Push to GitHub Container Registry:**
```bash
export GITHUB_USERNAME=behnamasadi
echo $GITHUB_TOKEN | docker login ghcr.io -u $GITHUB_USERNAME --password-stdin
docker tag kaggle-universal:latest ghcr.io/$GITHUB_USERNAME/kaggle-universal:latest
docker push ghcr.io/$GITHUB_USERNAME/kaggle-universal:latest
```

**4. Push to Docker Hub (alternative):**
```bash
docker login
docker tag kaggle-universal:latest <dockerhub-username>/kaggle-universal:latest
docker push <dockerhub-username>/kaggle-universal:latest
```

**5. Use on RunPod:** Configure pod with image `ghcr.io/<github-username>/kaggle-universal:latest` and mount projects to `/workspace/host`.

This setup gives you a single, reusable image that matches Kaggle's stack and works across all Kaggle challenges.

## 2. Kaggle Hardware and GPU

### Available GPUs

| GPU | VRAM | Compute Capability | Status |
|-----|------|-------------------|--------|
| **NVIDIA Tesla P100** | 16 GB | 6.0 (`sm_60`, Pascal) | ⚠️ **broken with current torch — see warning** |
| **NVIDIA Tesla T4** | 16 GB | 7.5 (`sm_75`, Turing) | ✅ works; also faster for FP16 |

You cannot choose which GPU you get. Kaggle assigns P100 or T4 randomly. Both have 16GB VRAM.

> ### ⚠️ CRITICAL (verified 2026-07-18): the P100 no longer works with Kaggle's torch
> Kaggle's preinstalled **torch 2.10.0+cu128** is built for archs `[sm_70, 75, 80, 86, 90, 100, 120]`
> — **`sm_60` (P100 Pascal) is NOT included.** So on a P100, any CUDA op fails with:
> ```
> CUDA error: no kernel image is available for execution on the device  (cudaErrorNoKernelImageForDevice)
> "Tesla P100 with CUDA capability sm_60 is not compatible with the current PyTorch installation"
> ```
> **Implications:**
> - If Kaggle assigns you a **P100, the GPU is unusable** with the stock torch → you must fall back to
>   CPU, request a **T4** instead, or bundle/install a Pascal-compatible (older) torch wheel offline.
> - **Always guard your code** with a CPU fallback (below) so a P100 assignment doesn't crash a
>   submission — this bit us on the Image Matching Challenge inference notebook.

**Check your GPU AND whether torch can actually use it:**
```python
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)              # e.g. (6, 0) = P100, (7, 5) = T4
    archs = torch.cuda.get_arch_list()                     # what THIS torch was compiled for
    usable = f"sm_{cap[0]}{cap[1]}" in archs
    print(f"GPU: {name} | capability sm_{cap[0]}{cap[1]} | torch archs: {archs}")
    print("USABLE:", usable)
    if not usable:
        print("⚠️ This GPU's arch is not in torch's build — expect cudaErrorNoKernelImageForDevice. Use CPU or a T4.")
else:
    print("No GPU available")
```

**Robust device selection (P100-safe) — verify a real kernel runs, else fall back to CPU:**
```python
import torch, torch.nn as nn
def safe_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        _ = nn.Conv2d(3, 4, 3).cuda()(torch.randn(1, 3, 8, 8, device="cuda"))
        torch.cuda.synchronize()
        return torch.device("cuda")
    except Exception as e:
        print("GPU unusable, using CPU:", str(e)[:80])
        return torch.device("cpu")
DEVICE = safe_device()
```

### GPU Comparison

| Aspect | P100 | T4 |
|--------|------|-----|
| **Compatibility w/ current torch** | ❌ **broken (sm_60 dropped)** | ✅ works |
| **FP32 performance** | 9.3 TFLOPS | 8.1 TFLOPS |
| **FP16 performance** | 18.7 TFLOPS | 65 TFLOPS (3-4× faster) |
| **Memory bandwidth** | 732 GB/s | 320 GB/s |
| **Power** | 250W | 70W |

**Bottom line:** on paper the P100 has better FP32/bandwidth, but with Kaggle's current torch it is
**unusable** — prefer/force a **T4**, and always keep a CPU fallback. Use mixed precision (FP16) on T4.

### System Resources

| Resource | Limit |
|----------|-------|
| **GPU Time** | ~30 hours/week |
| **CPU Cores** | 4 cores |
| **RAM** | 16-30 GB |
| **Disk Space** | ~73 GB (temporary) |
| **Session Time** | 9-12 hours max |
| **Internet** | Enabled (disabled in competitions) |

### Critical Limitations

1. **P100 assignment breaks GPU code with current torch** (see warning above) — guard with a CPU fallback
2. 16 GB VRAM (most common bottleneck)
3. Session timeout (9-12 hours)
4. Disk space (~73 GB)
5. **No internet in code competitions** → bundle wheels + model weights as a Dataset and load offline

### GPU Optimization Best Practices

1. Use Mixed Precision Training (FP16) — on T4
2. Gradient Accumulation
3. Monitor GPU Memory
4. Find Optimal Batch Size Dynamically
5. Clear Unused Variables and Cache
6. Gradient Checkpointing
7. Use Efficient Data Loading
8. Reduce Model Precision Strategically
9. **GPU-Agnostic Code Pattern — include the P100-safe `safe_device()` fallback above**

## 3. Kaggle Authentication

You need Kaggle API credentials to download datasets.

### Option 1: Local Development (kaggle.json)

1. Go to [Kaggle Account Settings](https://www.kaggle.com/settings/account)
2. Scroll to "API" section
3. Click "Create New Token"
4. Download `kaggle.json`
5. Place it in the correct location:

**Linux/macOS:**
```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json
```

**Windows:**
```powershell
# Place kaggle.json in:
C:\Users\<YourUsername>\.kaggle\kaggle.json
```

### Option 2: RunPod/Docker (Environment Variables)

```bash
export KAGGLE_USERNAME="your_username"
export KAGGLE_KEY="your_api_key"
```

Or in Dockerfile:
```dockerfile
ENV KAGGLE_USERNAME=your_username
ENV KAGGLE_KEY=your_api_key
```

### kaggle.json Locations Reference

| Platform | Location |
|----------|----------|
| **Linux** | `~/.kaggle/kaggle.json` or `~/.config/kaggle/kaggle.json` |
| **macOS** | `~/.kaggle/kaggle.json` |
| **Windows** | `C:\Users\<YourUsername>\.kaggle\kaggle.json` |
| **WSL** | `/home/<username>/.kaggle/kaggle.json` |

## Download Data in Notebook


```python
import kagglehub
import os
from pathlib import Path
import numpy as np
import warnings

def download_lungs_dataset():
    """
    Download dataset and find train/val/test directories.

    """
    # Download dataset
    dataset_path = kagglehub.dataset_download("omkarmanohardalvi/lungs-disease-dataset-4-types")
    print(f"✅ Dataset downloaded to: {dataset_path}")
    
    dataset_path = Path(dataset_path)
    
    # Look for train/val/test directories
    possible_train_dirs = list(dataset_path.rglob("train"))
    possible_val_dirs = list(dataset_path.rglob("val"))
    possible_test_dirs = list(dataset_path.rglob("test"))
    
    train_path = possible_train_dirs[0] if possible_train_dirs else None
    val_path = possible_val_dirs[0] if possible_val_dirs else None
    test_path = possible_test_dirs[0] if possible_test_dirs else None
    
    if train_path and val_path:
        print(f"✅ Found train: {train_path}")
        print(f"✅ Found val: {val_path}")
        if test_path:
            print(f"✅ Found test: {test_path}")
        return str(train_path), str(val_path), str(test_path) if test_path else None
    else:
        raise RuntimeError(f"Could not find train/val directories in {dataset_path}")
```

## 4. Kaggle CLI

For local development or custom environments, install the Kaggle API and kagglehub:

```bash
pip install kaggle kagglehub pyyaml
```

- **kaggle**: Official Kaggle API for datasets, competitions, notebooks (kernels)
- **kagglehub**: Modern dataset downloader (recommended over legacy kaggle CLI for downloads)
- **pyyaml**: For config files

The CLI can do the **entire loop from the terminal** — no browser, no copy-paste. This replaces
the old habit of "run the Kaggle Docker, experiment, then paste code into the web notebook and click
Submit." Same result, but reproducible, diffable, and scriptable.

---

### 4.1 Mental model — 4 object types

| Object | What it is | CLI group |
|---|---|---|
| **Competition** | the contest + its data + the leaderboard | `kaggle competitions` |
| **Dataset** | a versioned bundle of files (weights, wheels, CSVs) you upload | `kaggle datasets` |
| **Kernel** | a notebook/script that **runs on Kaggle's servers** (a.k.a. "notebook") | `kaggle kernels` |
| **Model** | pretrained-model hosting (rarely needed) | `kaggle models` |

**"kernel" = "notebook".** Same thing; "kernel" is just the API's name. A kernel has code +
metadata + attached inputs (datasets, competition data, GPU flag).

---

### 4.2 The crucial fork: CSV submission vs CODE competition

**(A) Simple / CSV competition** — compute predictions **anywhere** and upload a CSV:
```bash
kaggle competitions submit -c <comp-slug> -f submission.csv -m "my message"
```

**(B) Code competition**  ← *Image Matching, and most CV comps, are this kind* — you **cannot**
upload a bare CSV. You submit a **kernel**, and **Kaggle re-runs your notebook on a hidden test set**
(usually **no internet**, a GPU/CPU quota, and a **9–12 h time limit**). Your notebook must write
`/kaggle/working/submission.csv`.

> This is why "it ran fine for me" isn't enough: the score comes from Kaggle **re-executing** your
> code on data you never see. If that rerun is too slow it never scores (stuck `PENDING`). *(This bit
> us on IMC — exhaustive matching timed out; the fix was retrieval-restricted pairing + a wall-clock
> budget that flushes `submission.csv` incrementally.)*

Know which kind: read the comp's rules page — "Notebooks only" / "Code Competition" → type B.

---

### 4.3 The code-competition loop (what we do for IMC)

```
 develop locally ─▶ bundle offline deps as a Dataset ─▶ push Kernel ─▶
 Kernel runs on Kaggle ─▶ submit that Kernel version ─▶ Kaggle reruns on hidden test ─▶ score
```

**Step 1 — bundle offline deps as a Dataset.** The rerun has no internet, so anything `pip install`
or `torch.hub` would fetch must be pre-uploaded. A dataset folder just needs a `dataset-metadata.json`:
```json
{ "title": "imc 2026 deps min", "id": "asadibehnam/imc-2026-deps-min", "licenses": [{"name": "CC0-1.0"}] }
```
```bash
kaggle datasets create  -p imc_deps_min/                          # first time
kaggle datasets version -p imc_deps_min/ -m "add pycolmap wheel"  # update later
```
Put wheels (`pycolmap*.whl`), model weights (`*.pth`), etc. in that folder. (This is also how we
ship the **P100 torch fix** from Section 2 offline — as the dataset `asadibehnam/imc-torch251-cu121`.)

**Step 2 — describe the kernel in `kernel-metadata.json`** (next to the notebook):
```json
{
  "id": "asadibehnam/imc-2026-baseline-disk-lightglue-colmap",
  "code_file": "imc_infer.ipynb",
  "kernel_type": "notebook",
  "language": "python",
  "enable_gpu": true,
  "enable_internet": false,
  "dataset_sources": ["asadibehnam/imc-2026-deps-min", "asadibehnam/imc-torch251-cu121"],
  "competition_sources": ["image-matching-challenge-2025-ongoing"]
}
```
- `enable_internet:false` → forces you to use attached datasets (mirrors the scoring env).
- `dataset_sources` → mounted at `/kaggle/input/<dataset-name>/`.
- `competition_sources` → test data mounted at `/kaggle/input/<comp-slug>/`.

**Step 3 — push, watch, submit:**
```bash
cd competitions/image_matching_2026/submission

kaggle kernels push                          # upload code+metadata → Kaggle starts running it
kaggle kernels status <owner>/<kernel>       # QUEUED → RUNNING → COMPLETE / ERROR
kaggle kernels output <owner>/<kernel> -p ./out    # pull logs + the submission.csv it produced

# once COMPLETE, submit THAT kernel version to the competition:
kaggle competitions submit -c <comp-slug> \
   -k <owner>/<kernel> -v <version> -f submission.csv -m "message"
```
`-k` + `-v` is what makes it a *code* submission (points at the kernel version, not a local file).
`<version>` increments on every `kaggle kernels push` — the push output prints it ("Kernel version 9
successfully pushed").

**Step 4 — check the score / leaderboard:**
```bash
kaggle competitions submissions -c <comp-slug>       # your submissions + status + publicScore
kaggle competitions leaderboard <comp-slug> -s       # top of the public leaderboard
kaggle competitions leaderboard <comp-slug> --download   # full board as CSV (benchmarking)
```
Status: `PENDING` = still re-running (or queued) · `COMPLETE` = scored (`publicScore` filled) ·
`ERROR` = the rerun crashed/timed out (read the kernel log).

> ⚠️ **`kaggle kernels status` = COMPLETE does NOT mean your submission scored.** That status is the
> *interactive* run on the tiny public sample. The *submission* is a **separate** rerun on the full
> hidden test. A kernel can be COMPLETE while the submission is still PENDING or later ERRORs on
> timeout. **Always confirm with `kaggle competitions submissions`.**

---

### 4.4 Learn from other people's notebooks

The CLI is also how you study a competition — a public baseline may already beat you:
```bash
kaggle kernels list --competition <comp-slug> --sort-by voteCount   # best public notebooks
kaggle kernels pull <owner>/<notebook> -p ./their_nb                # download their code
kaggle competitions leaderboard <comp-slug> --download              # who scores what
```

---

### 4.5 Your old workflow vs this one

| Step | Old (browser) | CLI |
|---|---|---|
| Experiment | Kaggle Docker + mounted data, by hand | same locally, or `kaggle kernels push` to run on Kaggle GPU |
| Move code to notebook | copy-paste into the web editor | `code_file` in `kernel-metadata.json` + `kaggle kernels push` |
| Attach weights/wheels | upload via web | `kaggle datasets create` / `version` |
| Submit | click *Submit* | `kaggle competitions submit -k … -v …` |
| See score | refresh the page | `kaggle competitions submissions` |

**Tip:** develop the heavy pipeline as a normal `.py` (we do — `run_full.py`), then keep the
submission notebook as a thin wrapper that imports the validated pieces. Our
`submission/imc_infer.py` *is* the single notebook cell; we regenerate the `.ipynb` from it so the
code stays in version control, not trapped in a browser tab.

---

### 4.6 IMC quick reference (copy-paste)

```bash
SLUG=image-matching-challenge-2025-ongoing
K=asadibehnam/imc-2026-baseline-disk-lightglue-colmap

# study the field
kaggle competitions leaderboard $SLUG --download
kaggle kernels list --competition $SLUG --sort-by voteCount

# update offline deps (no internet at scoring time)
kaggle datasets version -p competitions/image_matching_2026/imc_deps_min -m "update"

# ship a new attempt
cd competitions/image_matching_2026/submission
kaggle kernels push
kaggle kernels status $K                       # wait for COMPLETE
kaggle competitions submit -c $SLUG -k $K -v <N> -f submission.csv -m "what changed"

# did it actually score?
kaggle competitions submissions -c $SLUG
```

---


## 5. Configuration

Edit `config/train.yaml`:

```yaml
data:
  kaggle_dataset: "owner/dataset-name"  # e.g., masoudnickparvar/brain-tumor-mri-dataset
  path: "./data/train"                   # Relative to kaggle_structure directory
```

## 6. Finding Kaggle Datasets

### Method 1: Kaggle Website

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets)
2. Search with keywords:
   - **Medical:** `brain`, `tumor`, `MRI`, `liver`, `kidneys`, `heart`, `lungs`, `vessel`, `lesion`, `COVID`, `lung`, `opacities`, `breast`, `cancer`, `Ultrasound`, `histopathology`, `segmentation`, `medical imaging`, `retinal`, `OCT`, `tomography`, `endoscopy`
   - **Computer Vision:** `object detection`, `segmentation`, `saliency`, `image similarity`
   - **3D Vision:** `LiDAR`, `SLAM`, `Photogrammetry`, `stereo`, `monocular depth`
3. Dataset name is in the URL: `kaggle.com/datasets/OWNER/DATASET-NAME`
4. Use `OWNER/DATASET-NAME` in `config/train.yaml`

### Method 2: Kaggle CLI

**Search:**
```bash
kaggle datasets list -s "MRI"
kaggle datasets list -s "MRI" --sort-by votes
kaggle datasets list -s "medical imaging" --sort-by votes -p 3
kaggle datasets list -s "segmentation" --max-size 50
```

Sort options: `hottest`, `votes`, `updated`, `active`, `published`

**Download:**
```bash
kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
```

### Method 3: Browse by Category

- [Sports](https://www.kaggle.com/datasets?topic=sports)
- [Medicine](https://www.kaggle.com/datasets?topic=health)
- [Computer Vision](https://www.kaggle.com/datasets?topic=computer-vision)

## 6. Kaggle Dataset Downloader (kagglehub)

This script creates symlinks from your project data directory to the Kaggle cache.

**What it does:**
1. Reads `config/train.yaml`
2. Downloads dataset via kagglehub (cache: `~/.cache/kagglehub/`)
3. Creates symlink from `kaggle_structure/data/train` to the downloaded dataset
4. Works locally (kaggle.json) and on RunPod (KAGGLE_USERNAME, KAGGLE_KEY)

[Downloader Script](scripts/kaggle_dataset_downloader.py)

## 7. Uploading and Creating Your Own Dataset

1. Go to [Kaggle Datasets](https://www.kaggle.com/datasets) and click **New Dataset**
2. Upload files or connect to cloud storage
3. Add title, description, and metadata
4. Use the [Kaggle API](https://www.kaggle.com/docs/api) for programmatic uploads

**References:**
- [Kaggle API documentation](https://www.kaggle.com/docs/api#installation)
- [RVP Group SLAM dataset](https://rvp-group.net/slam-dataset.html) (example)

## 8. Medal Targets & Architectures

Curated shortlist of Kaggle problems worth attacking for **medals**, drawn from the keyword
categories in Section 6 (Medical / Computer Vision / 3D Vision).

> **Medal rule:** Medals are awarded **only** in *Featured* and *Research* competitions.
> *Community* (reward = Kudos), *Playground*, and *Getting Started* competitions award **no medals**.

### Standard stack (same skeleton for every problem)

| Layer | Tool | Notes |
|---|---|---|
| Training loop | **PyTorch Lightning** | `LightningModule` + `Trainer(precision="16-mixed")`, callbacks: `ModelCheckpoint`, `EarlyStopping`, `SWA` |
| Backbones / weights | **timm** | `timm.create_model(name, pretrained=True, num_classes=N)`; timm encoders feed `segmentation_models_pytorch` for seg |
| 3D medical | **MONAI** / **nnU-Net** | `SegResNet`, `SwinUNETR`, `UNETR`, `Auto3DSeg` |
| Explainability | **pytorch-grad-cam** + **Captum** | Grad-CAM / Grad-CAM++ / EigenCAM + Integrated Gradients / saliency |
| 3D / volumes | **napari**, ITK-SNAP / 3D Slicer | interactive inspection of MRI / cell volumes |
| Augmentation | **albumentations** | preview + strong aug |
| Tracking | **Weights & Biases** / TensorBoard | loss curves + logged sample predictions |

**Winning recipe everywhere:** pretrained backbone → strong aug → k-fold CV → TTA → small ensemble.

### Live medal landscape — verified via Kaggle API 2026-07-18

The **only** currently-active, medal-eligible (Featured/Research) competition matching the
Medical/CV/3D domains is **Biohub – Cell Tracking** (Research, $60k, closes 2026-09-29). Other
active Featured comps (ARC Prize reasoning, ROGII wellbore/tabular, Pokémon TCG agent) are off-domain.
Classic medical/CV medal comps (ISIC-2024, Vesuvius, IMC-2025 Research) are **closed**.

### Active build plan (chosen 3)

- **A ⭐ Biohub – Cell Tracking** *(primary medal target)* — Research, live, ~2.5 mo. 3D+t `.zarr`
  microscopy volumes. MONAI `SegResNet`/StarDist-3D segmentation → graph/nearest-neighbor tracker.
  **Requires joining + accepting rules.**
- **B  Image Matching Challenge 2026-ongoing** *(practice / COLMAP edge — ⚠️ Community, NO medals)* —
  SfM pipeline: retrieval → deep matching (LightGlue/LoFTR) → RANSAC → `pycolmap`. Live leaderboard +
  strong writeup, but not a medal path.
- **C  APTOS Diabetic Retinopathy** *(reference pipeline / skill build — closed comp)* — the reusable
  Lightning + timm + Grad-CAM template; QWK ≈ 0.90+ reproducible.

### The 10-candidate reference table

| # | Problem | Category | Task | Architecture | Status (2026-07-18) |
|---|---|---|---|---|---|
| 1 | **Biohub – Cell Tracking** | Medical · 3D microscopy | 3D segmentation + tracking | StarDist-3D / 3D U-Net (MONAI) + graph tracker | 🟢 **LIVE · Research · medals** |
| 2 | Image Matching Challenge 2026-ongoing | 3D Vision · SfM | feature match → camera pose | ALIKED/DISK + LightGlue or LoFTR → `pycolmap` | 🟡 LIVE · **Community (no medals)** |
| 3 | ARC Prize 2026 (AGI-2/3) | Reasoning | abstract reasoning | LLM / program synthesis | 🟢 LIVE · Featured (off-domain, brutal) |
| 4 | RSNA Intracranial Aneurysm | Medical · vessel/brain 3D | detect + localize | 2.5D EfficientNetV2 (timm) / 3D CNN | ⚪ verify — not in active list |
| 5 | HuBMAP Kidney/Vessel | Medical · kidneys/vessel | segmentation | U-Net++ / SegFormer + timm encoder | ⚪ recurring |
| 6 | PANDA Prostate Grading | Medical · histopathology | ordinal WSI grading | timm EfficientNet + MIL | 🔴 closed (practice) |
| 7 | APTOS Diabetic Retinopathy | Medical · retinal/OCT | ordinal 5-class | EfficientNet-B4/B5 (timm) + CLAHE + Grad-CAM | 🔴 closed (practice) |
| 8 | SIIM-ACR Pneumothorax | Medical · lungs | 2D segmentation | U-Net++ + timm encoder | 🔴 closed (practice) |
| 9 | Kvasir-SEG Polyp | Medical · endoscopy | 2D segmentation | SegFormer / U-Net++ + timm encoder | ⚪ dataset |
| 10 | ISIC-2024 Skin Cancer | Medical · lesion | classification | timm EfficientNet + metadata + MIL | 🔴 closed (strong practice) |

**Gates (only the user can clear):** (1) Kaggle API token at `~/.kaggle/kaggle.json` ✅ done;
(2) **join Biohub + accept rules** on the website; (3) live medal comps are often *code competitions*
— develop locally, train weights on the local GPU, submit an inference-only offline notebook.

Project code lives under [`competitions/`](competitions/).
